# Пример 04. Свойства матричного умножения

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** умножение матриц, некоммутативность, согласованные размерности; связь со свёрточными слоями в нейросетях.

## Условие

Требуется вычислить произведения нескольких пар матриц $AB$ и $BA$, обращая внимание на размерности и на то, имеет ли смысл обратное произведение в каждом случае. Отдельно рассматривается применение матричных операций к небольшому изображению через свёрточный слой.

## Математическая идея

Произведение $C = AB$ матриц $A\in\mathbb{R}^{m\times n}$ и $B\in\mathbb{R}^{n\times p}$ определено только если число столбцов $A$ равно числу строк $B$, и результат $C$ имеет размерность $m\times p$. Элементы вычисляются по формуле

$$C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}.$$

Важные свойства:

- $AB \ne BA$ в общем случае (некоммутативность);
- если $A\in\mathbb{R}^{m\times n}$, $B\in\mathbb{R}^{n\times p}$, то $BA$ определено только при $m = p$;
- даже если оба произведения существуют, их размерности и значения могут различаться.

## Решение

1. Вычисляются произведения для пар матриц $3\times 3$ в обоих порядках — подтверждается некоммутативность.
2. Берутся матрицы $A\in\mathbb{R}^{2\times 4}$ и $B\in\mathbb{R}^{4\times 2}$; вычисляется $AB\in\mathbb{R}^{2\times 2}$. Обратное произведение $BA\in\mathbb{R}^{4\times 4}$ также определено, но имеет другую размерность.
3. Меняется порядок матриц — проверяется, что результат отличается.

## Реализация на Python

Используется оператор `@` для матричного умножения из NumPy. Каждый блок кода работает с конкретными матрицами, заданными через `np.array`. В последней части применяется `torch.nn.Conv2d` с фиксированным ядром `[[1,0],[0,1]]`, чтобы показать, как матричная операция (свёртка) преобразует входной «образ» $4\times 4$ в карту признаков $3\times 3$.


In [1]:
import numpy as np

A = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
B = np.array([[1, 1, 0], [0, 1, 1], [1, 0, 1]])
print(A @ B)

[[ 4  3  5]
 [10  9 11]
 [16 15 17]]


In [2]:
A = np.array([[1, 1, 0], [0, 1, 1], [1, 0, 1]])
B = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(A @ B)

[[ 5  7  9]
 [11 13 15]
 [ 8 10 12]]


In [4]:
A = np.array([[1, 2, 1, 2], [4, 1, -1, -4]])
B = np.array([[0, 3], [1, -1], [2, 1], [5, 2]])
print(A @ B)

[[ 14   6]
 [-21   2]]


In [5]:
A = np.array([[0, 3], [1, -1], [2, 1], [5, 2]])
B = np.array([[1, 2, 1, 2], [4, 1, -1, -4]])
print(A @ B)

[[ 12   3  -3 -12]
 [ -3   1   2   6]
 [  6   5   1   0]
 [ 13  12   3   2]]


## Дополнительный пример

**Идея.** Свёрточный слой можно рассматривать как специальную форму умножения матриц с разделением весов: ядро $K\in\mathbb{R}^{k\times k}$ «скользит» по входному изображению и в каждой позиции вычисляет сумму поэлементных произведений. Это эквивалентно умножению входа на разреженную матрицу с повторяющимися блоками.

**Что демонстрирует код.** Создаётся входной тензор $1\times 1\times 4\times 4$ и применяется свёртка с ядром $2\times 2$, равным единичной матрице. Результат — карта признаков $3\times 3$, каждый элемент которой равен сумме диагональных элементов соответствующего окна входа.


In [8]:
import torch
import torch.nn as nn

input = torch.tensor([[[[1, 2, 3, 4],
                      [5, 6, 7, 8],
                      [9, 10, 11, 12],
                      [13, 14, 15, 16]]]], dtype=torch.float32)

conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, bias=False)
conv.weight.data = torch.tensor([[[[1, 0], [0, 1]]]], dtype=torch.float32)

output = conv(input)
print(output)

tensor([[[[ 7.,  9., 11.],
          [15., 17., 19.],
          [23., 25., 27.]]]], grad_fn=<ConvolutionBackward0>)


## Проверка результата

Корректность умножения проверяется выводом результатов и контролем размерностей: для матриц $2\times 4$ и $4\times 2$ произведения имеют размерности $2\times 2$ и $4\times 4$ соответственно. Корректность свёртки проверяется тем, что размер выхода $3\times 3$ соответствует формуле $(H - k + 1)\times (W - k + 1)$ при отсутствии паддинга и стрида $1$.

## Вывод

Матричное умножение — базовая операция, на которой строятся практически все модели машинного обучения. Понимание его свойств (некоммутативность, согласованность размерностей) необходимо для работы с линейными слоями, свёртками и механизмами внимания. Свёрточный слой — это эффективная структура, эквивалентная умножению на разреженную матрицу с повторяющимися весами.
